In [ ]:
import datasets
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import nltk
from sentence_transformers import SentenceTransformer, util
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments
from torch.utils.data import Subset
from tqdm import tqdm
import os
import random


2026-02-08 16:41:17.149138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770568877.332410      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770568877.388399      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770568877.828204      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770568877.828255      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770568877.828258      24 computation_placer.cc:177] computation placer alr

In [2]:
dataset = load_dataset("IlyaGusev/gazeta")

README.md: 0.00B [00:00, ?B/s]

default/train/0000.parquet:   0%|          | 0.00/252M [00:00<?, ?B/s]

default/train/0001.parquet:   0%|          | 0.00/22.7M [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/27.8M [00:00<?, ?B/s]

default/test/0000.parquet:   0%|          | 0.00/30.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60964 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6369 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6793 [00:00<?, ? examples/s]

In [ ]:
def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random seed set to {seed}")

set_seed(42)

Random seed set to 42


In [4]:
model_name = "google/mt5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
bert_model = SentenceTransformer("ai-forever/sbert_large_mt_nlu_ru").to(device)
class SummaryDataset(Dataset):
    def __init__(self, dataset, sentences_distribution, max_input_len, device):
        """
        Args:
            dataset: Original dataset with 'text' and 'summary' fields.
            sentences_distribution (dict): Dictionary specifying how many sentences to select 
                                           from text for different text length categories.
            max_input_len (int): Maximum token length for tokenizer.
            device (str): Device to run embeddings and tensors on
        """
        super().__init__()
        
        self.dataset = dataset
        self.device = device
        self.tokenizer = tokenizer
        self.tokenizer.truncation_side = 'right'
        
        self.sentences_distribution = sentences_distribution
        self.max_input_len = max_input_len
        
    def __len__(self):
        return len(self.dataset)

    
    def _sentence_emb(self, sentence):
        """
        Compute sentence embeddings using the pre-loaded BERT model.
        Args:
            sentence (str or List[str]): Sentence or list of sentences to embed.
        Returns:
            torch.Tensor: Embedding tensor(s) for the input sentence(s).
        """
        with torch.no_grad():
            return bert_model.encode(sentence, convert_to_tensor=True)
    
    def _devide_text_on_chunks(self, text, summary_emb, sent_in_chunk):
        """
        Split text into chunks and compute similarity of each sentence with the summary embedding.
        Args:
            text (str): Full input text.
            summary_emb (torch.Tensor): Embedding vector of the target summary.
            sent_in_chunk (int): Number of sentences per chunk.
        Returns:
            List[List[Tuple[int, str, float]]]: Each chunk is a list of tuples:
                (original_sentence_index, sentence_text, similarity_score)
        """
        sentences = nltk.sent_tokenize(text)
        chunks = [[]]

        for i, s in enumerate(sentences):
            if len(chunks[-1]) < sent_in_chunk:
                chunks[-1].append((i, s))
            else:
                chunks.append([(i, s)])

        scored_chunks = []
        for chunk in chunks:
            s_texts = [c[1] for c in chunk]
            emb_sents = self._sentence_emb(s_texts)  # (m, d)
            
            sims = util.cos_sim(emb_sents, summary_emb.to(self.device).unsqueeze(0)).squeeze(1)  # (m,)
            
            scored = []
            for (orig_idx, sent), score in zip(chunk, sims):
                scored.append((orig_idx, sent, float(score.item())))
            scored_chunks.append(scored)
        return scored_chunks

    def _get_need_sentences(self, text, summary_emb, sentences_distribution):
        """
        Select important sentences from the text based on similarity to summary embedding
        and the configuration in sentences_distribution.
        Args:
            text (str): Original text.
            summary_emb (torch.Tensor): Embedding of the target summary.
            sentences_distribution (dict): Dictionary specifying how many sentences to select:
                'best_sbert' - top most similar sentences
                'worst_sbert' - least similar sentences
                'random' - additional random sentences
        Returns:
            List[str]: List of selected sentences, sorted in original text order.
        """
        raw_chunks = self._devide_text_on_chunks(text, summary_emb, sentences_distribution['sent_in_chunk'])
        
        all_need_sentences = []
        for chunk in raw_chunks:
            chunk_sorted = sorted(chunk, key=lambda cur: cur[2], reverse = True)
        
            need_sentences = []
            best = sentences_distribution['best_sbert']
            worst = sentences_distribution['worst_sbert']
            rand_n = sentences_distribution['random']

            
            for i in range(min(len(chunk_sorted), best)):
                need_sentences.append(chunk_sorted[i])
            for i in range(max(0, min(len(chunk_sorted) - best, worst))):
                need_sentences.append(chunk_sorted[- i - 1])
                
            for _ in range(max(0, min(len(chunk_sorted) - best - worst, rand_n))):
                cur_idx = torch.randint(
                    low = best, 
                    high = len(chunk_sorted) - worst,
                    size=(1,)
                )[0].item()
                need_sentences.append(chunk_sorted[cur_idx])
            need_sentences = sorted(need_sentences, key = lambda cur: cur[0])
            need_sentences = [cur[1] for cur in need_sentences]
            
            all_need_sentences += need_sentences
            
        return all_need_sentences

    def _tokenize_text(self, text, add_special_tokens: bool):
        """
        Tokenize text using the tokenizer.
        Args:
            text (str): Text to tokenize.
            add_special_tokens (bool): Whether to include special tokens (BOS/EOS).
        Returns:
            Tuple[torch.Tensor, torch.Tensor]: input_ids and attention_mask tensors.
        """
        out_token = self.tokenizer(
            text,
            max_length=self.max_input_len,
            truncation=True,   
            padding=False,
            return_tensors='pt',
            add_special_tokens = add_special_tokens,
        )
        return out_token['input_ids'], out_token['attention_mask']
    
    def __getitem__(self, idx):
        """
        Retrieve a single dataset item, processing text and summary into input tensors.
        Steps:
            1. Determine text length category (small/medium/big).
            2. Compute embedding for target summary.
            3. Select relevant sentences from text.
            4. Build input prompt and ending tokens.
            5. Tokenize input and summary to generate tensors.
        Args:
            idx (int): Index of the sample in the dataset.
        Returns:
            dict: {
                'input_ids': tensor for model input,
                'attention_mask': tensor indicating valid tokens,
                'labels': tensor with token IDs of the target summary
            }
        """
        text = self.dataset[idx]['text']
        cur_len = len(self.tokenizer(text)['input_ids'])
        
        if (cur_len <= 1100):
            sentences_distribution = self.sentences_distribution['small']
        elif (cur_len <= 1600):
            sentences_distribution = self.sentences_distribution['med']
        else:
            sentences_distribution = self.sentences_distribution['big']
        summary = self.dataset[idx]['summary']
        
        summary_emb = self._sentence_emb(summary)
        
        need_sentences = self._get_need_sentences(text, summary_emb, sentences_distribution)
        need_sentences = " ".join(need_sentences)
        
        in_text = "Сформулируй краткое содержание:\n\n" + need_sentences        
        ending_text = "...\n\nКраткое содержание:" + self.tokenizer.eos_token
        
        in_text_ids, _ = self._tokenize_text(in_text, add_special_tokens = False)
        ending_ids, _ = self._tokenize_text(ending_text, add_special_tokens = False)
        in_text_ids = in_text_ids.squeeze(0)
        ending_ids = ending_ids.squeeze(0)
    
        input_ids = torch.cat((in_text_ids, ending_ids), dim=0)
        attn_mask = torch.ones_like(input_ids)
        
        labels, _ = self._tokenize_text(summary, add_special_tokens=True)
        labels = labels.squeeze(0)
        
        ans = {
            'input_ids': input_ids,
            'attention_mask': attn_mask,
            'labels': labels,
        }
        return ans



sentences_distribution = {
    'small': { # before 1100
        'sent_in_chunk': 11,
        'best_sbert': 2,
        'worst_sbert': 1,
        'random': 2,
    }, 
    'med': {# before 1600
        'sent_in_chunk': 15,
        'best_sbert': 2,
        'worst_sbert': 1,
        'random': 2,
    },
    'big': {
        'sent_in_chunk': 17,
        'best_sbert': 2,
        'worst_sbert': 1,
        'random': 2,
    }
}
max_input_len = 504
device = 'cuda'


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/866 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [6]:
train_dataset = SummaryDataset(dataset['train'], sentences_distribution, max_input_len, device)
valid_dataset = SummaryDataset(dataset['validation'], sentences_distribution, max_input_len, device)

In [7]:
valid_dataset_samples = []

for i in tqdm(range(len(valid_dataset))):
    cur = valid_dataset[i]
    valid_dataset_samples.append({
        "input_ids": cur["input_ids"].tolist(),
        "attention_mask": cur["attention_mask"].tolist(),
        "labels": cur["labels"].tolist(),
    })

100%|██████████| 6369/6369 [26:50<00:00,  3.95it/s]


In [8]:
train_dataset_samples = []

for i in tqdm(range(len(train_dataset))):
    cur = train_dataset[i]
    train_dataset_samples.append({
        "input_ids": cur["input_ids"].tolist(),
        "attention_mask": cur["attention_mask"].tolist(),
        "labels": cur["labels"].tolist(),
    })
    if (i >= 60001):
        break

 98%|█████████▊| 60001/60964 [4:33:28<04:23,  3.66it/s]


In [ ]:
train_dataset_hf = datasets.Dataset.from_list(train_dataset_samples)
valid_dataset_hf = datasets.Dataset.from_list(valid_dataset_samples)

valid_dataset_hf.save_to_disk("prepared/valid")
train_dataset_hf.save_to_disk("prepared/train")

Saving the dataset (0/1 shards):   0%|          | 0/6369 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/60002 [00:00<?, ? examples/s]